# 🌿 AgriVision: YOLO26 Plant Disease Segmentation & Detection Pipeline
### Flagship SOTA Model Training on Dual NVIDIA T4 GPUs
**Author:** AgriVision AI Team / Khang Phan Trọng  
**Model Architecture:** Ultralytics YOLO26 (NMS-Free Dual Head, No DFL for 43% CPU Inference Boost)  
**Task:** Instance Segmentation (`yolo26m-seg`) & Object Detection (`yolo26m`)  
**Hardware:** Kaggle Dual NVIDIA T4 (32GB VRAM total), 4 vCPUs, 30GB RAM


In [ ]:
# 1. Hardware Inspection: Check GPU availability and PyTorch CUDA backend
import torch
import sys
import os

print(f"Python Version: {sys.version}")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    device_count = torch.cuda.device_count()
    print(f"GPU Count: {device_count}")
    for i in range(device_count):
        print(f"  Device {i}: {torch.cuda.get_device_name(i)} (VRAM: {torch.cuda.get_device_properties(i).total_memory / 1e9:.2f} GB)")
else:
    print("WARNING: CUDA not detected! Ensure GPU accelerator is enabled in Kaggle notebook settings.")


In [ ]:
# 2. Install latest Ultralytics & Computer Vision libraries
!pip install -q -U ultralytics albumentations opencv-python-headless onnx onnxslim onnxruntime-gpu
import ultralytics
ultralytics.checks()


In [ ]:
# 3. Locate and configure Plant Pathology Dataset
import glob
import yaml
from pathlib import Path

# Check attached Kaggle datasets
kaggle_input_dir = Path("/kaggle/input")
datasets = list(kaggle_input_dir.glob("*"))
print("Found Kaggle Datasets:", [d.name for d in datasets])

# Locate PlantDoc or Plant Disease Dataset
data_yaml_candidates = list(kaggle_input_dir.glob("**/*data*.yaml")) + list(kaggle_input_dir.glob("**/*dataset*.yaml"))
selected_yaml = None

if data_yaml_candidates:
    selected_yaml = str(data_yaml_candidates[0])
    print(f"Found existing dataset configuration: {selected_yaml}")
else:
    print("Setting up unified PlantDoc & Vietnamese Crop Disease dataset...")
    # Create standard YOLO dataset configuration
    dataset_dir = Path("/kaggle/working/plant_dataset")
    dataset_dir.mkdir(parents=True, exist_ok=True)
    
    # Download sample benchmark dataset if not already mounted
    !git clone --depth 1 https://github.com/ultralytics/ultralytics.git /kaggle/working/ultralytics_repo


In [ ]:
# 4. Load Ultralytics YOLO26 Model
from ultralytics import YOLO

# Select YOLO26 model tier:
# yolo26n-seg: Nano (3M params - extreme edge/mobile)
# yolo26s-seg: Small (10M params)
# yolo26m-seg: Medium (22M params - recommended optimal tradeoff)
# yolo26l-seg: Large (45M params)
# yolo26x-seg: Extra Large (60M params)

MODEL_NAME = "yolo26m-seg.pt" # Ultralytics fallback auto-downloads weights or architecture definition
try:
    model = YOLO(MODEL_NAME)
    print(f"Successfully initialized {MODEL_NAME}")
except Exception as e:
    print(f"Fallback to latest segmentation model: {e}")
    model = YOLO("yolo11m-seg.pt")


In [ ]:
# 5. Execute Multi-GPU Distributed Training
# Parameters tuned specifically for Kaggle Dual T4 (16GB VRAM each)
devices = [0, 1] if torch.cuda.device_count() >= 2 else (0 if torch.cuda.is_available() else "cpu")

results = model.train(
    data="coco8-seg.yaml", # Default segmentation benchmark (swap with plant data.yaml)
    epochs=30,
    imgsz=640,
    batch=16,
    device=devices,
    workers=4,
    optimizer="AdamW",
    lr0=0.001,
    lrf=0.01,
    weight_decay=0.0005,
    warmup_epochs=3.0,
    warmup_momentum=0.8,
    amp=True, # Automatic Mixed Precision (fp16) for 2x faster training
    patience=10,
    save=True,
    save_period=5,
    project="/kaggle/working/agrivision_runs",
    name="yolo26_plant_disease",
    exist_ok=True,
    verbose=True
)
print("Training completed successfully!")


In [ ]:
# 6. Comprehensive Validation & Pathology Metrics
metrics = model.val()
print("\n--- Validation Results ---")
print(f"Mask mAP50: {metrics.seg.map50:.4f}")
print(f"Mask mAP50-95: {metrics.seg.map:.4f}")
print(f"Box mAP50: {metrics.box.map50:.4f}")
print(f"Box mAP50-95: {metrics.box.map:.4f}")


In [ ]:
# 7. Export Best Model to ONNX and Engine Formats
import shutil

best_weight = "/kaggle/working/agrivision_runs/yolo26_plant_disease/weights/best.pt"
if os.path.exists(best_weight):
    print(f"Exporting {best_weight} to ONNX format...")
    trained_model = YOLO(best_weight)
    onnx_path = trained_model.export(format="onnx", dynamic=True, simplify=True, opset=17)
    print(f"Exported ONNX: {onnx_path}")
    
    # Copy to root output directory for direct download
    shutil.copy2(best_weight, "/kaggle/working/yolo26_plant_disease_best.pt")
    if onnx_path and os.path.exists(str(onnx_path)):
        shutil.copy2(str(onnx_path), "/kaggle/working/yolo26_plant_disease_best.onnx")
    print("Files ready in /kaggle/working/ for direct download!")
